# GP_ELITE — try it in your browser

**GP_ELITE recovers interpretable physical laws from small experimental datasets.**

Declare your units and the search only ever builds dimensionally valid equations — a hard
constraint, not a soft penalty. Its operating envelope is measured, not claimed: how many
points it needs, how much noise it tolerates, and where it fails.

This notebook takes about **8 minutes**. No install, no account, nothing to configure.

1. Recover a law of physics from raw numbers (~30 s)
2. See what declaring units actually buys you (~25 s)
3. Watch it rediscover a law of physics from real 1933 measurements (~4 min)
4. **Run it on your own data**

Run each cell with `Shift+Enter`, or `Runtime > Run all`.

Repository: https://github.com/ariel95500-create/gp-elite

In [ ]:
#@title Install GP_ELITE (about 30 seconds)
!pip install --quiet git+https://github.com/ariel95500-create/gp-elite.git

import gp_elite, numpy as np
print("gp_elite", gp_elite.__version__, "| numpy", np.__version__)

## 1. Recover Coulomb's law from numbers alone

We generate 200 measurements of the force between two charges and hand GP_ELITE **only the
numbers** — four input columns and one output column. No formula, no hint, no template.

The law it has to find is $F = \dfrac{q_1 q_2}{4\pi\varepsilon r^2}$.

Watch the constant in the answer.

In [ ]:
import numpy as np, time
from gp_elite import symbolic_regression

rng = np.random.RandomState(0)
q1  = rng.uniform(1, 5, 200)
q2  = rng.uniform(1, 5, 200)
eps = rng.uniform(1, 3, 200)
r   = rng.uniform(1, 3, 200)

X = np.column_stack([q1, q2, eps, r])
y = q1 * q2 / (4 * np.pi * eps * r**2)          # the law we are hiding

t0 = time.time()
model = symbolic_regression(
    X, y,
    feature_names=["q1", "q2", "eps", "r"],
    operators="physical",
    normalize="none",     # these four columns share a comparable scale.
                          # Leave it out (default "auto") when your columns
                          # span very different magnitudes -- see section 3.
    generations=25, speed="fast", restarts=3, seed=0,
)

print(f"\nfound in {time.time()-t0:.0f} s")
print("equation :", model.expression)
print("1/(4*pi) =", round(1/(4*np.pi), 6), " <- compare with the constant above")

GP_ELITE was never told about $\pi$. It recovered `0.079577` — which is $1/(4\pi)$ — from
200 noisy-free measurements, together with the exact structure $q_1 q_2 / (\varepsilon r^2)$.

---

## 2. What declaring units buys you

This is where GP_ELITE differs from other genetic-programming engines. Elsewhere,
dimensional consistency is a **penalty added to the loss** — wrong equations are built, then
discouraged. Here it is a **construction rule**: crossover, mutation and the initial
population only ever produce dimensionally valid trees. Nothing invalid is ever built.

Same problem twice — torque, $\tau = r F \sin\theta$ — first blind, then with units declared.

In [ ]:
rng   = np.random.RandomState(13)
r_    = rng.uniform(1, 5, 200)
F_    = rng.uniform(1, 5, 200)
theta = rng.uniform(0, np.pi, 200)

X2 = np.column_stack([r_, F_, theta])
y2 = r_ * F_ * np.sin(theta)

def run(label, **extra):
    t0 = time.time()
    m = symbolic_regression(X2, y2, feature_names=["r", "F", "theta"],
                            operators="trig", normalize="none",
                            generations=25, speed="fast",
                            restarts=3, seed=0, **extra)
    err = np.mean((m.predict(X2) - y2)**2) / np.var(y2)
    print(f"{label:<22} {time.time()-t0:5.0f} s   error={err:.1e}   size={m.size}")
    print(f"{'':22} {m.expression}\n")

metre    = {"m": 1}
newton   = {"kg": 1, "m": 1, "s": -2}
radian   = {}                                  # an angle is dimensionless
joule    = {"kg": 1, "m": 2, "s": -2}

run("without units")
run("with units", units=[metre, newton, radian], target_units=joule)

---

## 3. Real data: Nikuradse's 1933 pipe-friction measurements

Everything above was synthetic. This is not.

In the early 1930s Johann Nikuradse glued sand grains of a known size inside pipes and measured
the friction of turbulent flow: **362 real measurements**, six roughness ratios, Reynolds numbers
from about 4,300 to 1,000,000. The dataset is still in use — the full functional dependence of
friction on Reynolds number and relative roughness is regarded as an open problem.

GP_ELITE gets the raw numbers and nothing else: relative roughness `r_k`, the base-10 log of the
Reynolds number, and the friction as `log10(100*lambda)`.

The reference to beat is the textbook **Prandtl–von Kármán** relation, which in this target space
reads `2 - 2*log10(2*log10(r/k) + 1.74)` — with **no fitted parameters at all**.

In [ ]:
import pandas as pd, numpy as np

URL = ("https://github.com/EpistasisLab/pmlb/raw/master/datasets/"
       "nikuradse_1/nikuradse_1.tsv.gz")
df = pd.read_csv(URL, sep="\t", compression="gzip")

Xn = df[["r_k", "log_Re"]].to_numpy(float)
yn = df["target"].to_numpy(float)
print(f"{len(yn)} real measurements, roughness ratios: "
      f"{sorted(df.r_k.unique())}")

def prandtl_von_karman(r_k):
    return 2.0 - 2.0 * np.log10(2.0 * np.log10(r_k) + 1.74)

r2_pvk = 1 - np.mean((prandtl_von_karman(Xn[:, 0]) - yn)**2) / np.var(yn)
print(f"Prandtl-von Karman, zero fitted parameters : R2 = {r2_pvk:.4f}")

Now the search. **This one takes a few minutes** — real data has no exact solution to stop at,
so the engine uses its whole budget.

With the full protocol (`generations=30, restarts=4`, roughly ten minutes) GP_ELITE returns a
seven-node expression, `1.32 - 0.56*log(log(r_k))`, whose R² is 0.9455 against the reference law's
0.9456 — a dead heat — and which tracks it with a correlation of 0.9992 across the six roughness
ratios. The shorter run below may land somewhere else; the comparison printed at the end tells you
honestly what it found, whatever that is.

In [ ]:
import time
from gp_elite import symbolic_regression

t0 = time.time()
m = symbolic_regression(Xn, yn, feature_names=["r_k", "log_Re"],
                        operators="physical", normalize="none",
                        generations=30, speed="fast", restarts=2, seed=0)
print(f"searched for {time.time()-t0:.0f} s\n")

def r2(e):
    return 1 - np.mean((e.predict(Xn) - yn)**2) / np.var(yn)

cands = sorted({(int(e.size), round(r2(e), 6), e.expression)
                for e in list(m.pareto or []) + [m]})
print("size   R2      expression")
for s, r, ex in cands:
    print(f"{s:>4}  {r:>6.4f}  {ex[:78]}")

# the most compact expression that still explains 90% of the variance
good = [c for c in cands if c[1] > 0.90]
if good:
    s, r, ex = good[0]
    six = np.array(sorted(df.r_k.unique()))
    grid = np.column_stack([six, np.full(len(six), Xn[:, 1].mean())])
    best = min((e for e in list(m.pareto or []) + [m]),
               key=lambda e: abs(r2(e) - r) + abs(e.size - s))
    pred, theory = best.predict(grid), prandtl_von_karman(six)
    print(f"\nMost compact good expression ({s} nodes, R2={r:.4f}):\n  {ex}")
    print(f"\n{'r/k':>7}{'Prandtl-von Karman':>21}{'GP_ELITE':>12}")
    for a, b, c in zip(six, theory, pred):
        print(f"{a:>7.1f}{b:>21.4f}{c:>12.4f}")
    print(f"\ncorrelation with the textbook law: "
          f"{np.corrcoef(theory, pred)[0,1]:.4f}")
else:
    print("\nNo compact expression above R2=0.90 in this short run -- "
          "see benchmarks/real_nikuradse.py in the repo for the full protocol.")

### What this does and does not show

GP_ELITE **rediscovers** a known law here; it does not beat it. On a roughness ratio removed from
training entirely, the textbook relation predicts the level to within 0.04 standard deviations of
the curve, against 0.25 for the recovered seven-node form — a systematic 2.6% approximation gap.

That is the honest summary, and it is the point: the engine found the right *family* of functions
from raw measurements, with constants close enough to be useful and far enough to be visible.
Full protocol, held-out-roughness test and telemetry: `benchmarks/real_nikuradse.py`.

Declaring units is a few short dictionaries. In exchange the search space collapses to the
physically possible: on this run the constrained search is about **six times faster** and
returns the textbook form `sin(theta) * r * F`, while the blind search spends longer and
wraps its answer in a spurious `abs()`. On a benchmark of 15 Feynman equations, the
dimensionally constrained arm returned the textbook form of *every* equation it recovered.

Both are correct here — the point is the cost and the shape of the answer. On harder
targets the blind search stops being correct at all.

You never have to use units — leave them out and GP_ELITE works as a general-purpose
symbolic regressor.

---

## 4. Your turn — run it on your own data

**This is the part we care about.** GP_ELITE is young and has had almost no outside users.
Whatever happens below — a good result, a wrong one, a crash — is useful to us.

Two ways in. Use either cell.

### Option A — upload a CSV
One column per variable, **the quantity to explain in the last column**, one header row.

In [ ]:
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
except ImportError:
    df = pd.read_csv("your_file.csv")          # outside Colab: set the path

names = list(df.columns[:-1])
Xu    = df[names].to_numpy(dtype=float)
yu    = df[df.columns[-1]].to_numpy(dtype=float)
print(f"{Xu.shape[0]} rows, {Xu.shape[1]} inputs {names} -> {df.columns[-1]}")

model = symbolic_regression(
    Xu, yu, feature_names=names,
    operators="physical",      # "trig" if you expect sines/cosines, "poly" for polynomials
    generations=30, speed="fast", restarts=4, seed=0,
)
print("\nequation:", model.expression)

print("\nsimpler alternatives found along the way:")
for e in (model.pareto or [])[:6]:
    print(f"  size {e.size:>3} : {e.expression}")

### Option B — paste your numbers
No file needed: replace the three lists below with your own measurements.

In [ ]:
# --- replace with your own measurements -------------------------------
pressure    = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5]
volume      = [2.0, 2.0, 2.0, 3.0, 3.0, 3.0, 4.0, 4.0, 5.0, 5.0]
energy      = [3.0, 4.5, 6.0, 11.25, 13.5, 15.75, 24.0, 27.0, 37.5, 41.25]   # to explain
# ----------------------------------------------------------------------

Xu = np.column_stack([pressure, volume])
yu = np.asarray(energy, dtype=float)

model = symbolic_regression(Xu, yu, feature_names=["pressure", "volume"],
                            operators="physical", generations=30,
                            speed="fast", restarts=4, seed=0)
print("\nequation:", model.expression)

---

## Tell us what happened

One sentence is enough, and a disappointing result is worth more to us than a good one.

**→ https://github.com/ariel95500-create/gp-elite/issues**

Useful to mention, if you can:

* how many rows and columns, and roughly what the data is
* what you hoped to get, and what came out
* how long it took, and whether you used `units=`

You do not need to share your data.

### If the result was poor, three things usually help

| symptom | try |
| --- | --- |
| equation is huge and unreadable | raise `restarts` to 8, or use `units=` |
| missing a sine or cosine | `operators="trig"` |
| columns span very different magnitudes | `normalize="auto"` (the default) |
| columns are comparable, results look off | `normalize="none"` |

### Known limits, measured rather than guessed

* nested rational forms such as $(u+v)/(1+uv/c^2)$ are the hardest case and often fail
* a handful of wildly wrong points (outliers) hurt far more than uniform measurement noise

Full benchmarks and telemetry live in the repository.